In [ ]:
!apt-get update -qq
!apt-get install -y -qq ffmpeg

!pip install -q yt-dlp
!pip install -q openai-whisper
!pip install -q "scenedetect[opencv]"
!pip install -q opencv-python
!pip install -q transformers
!pip install -q torch torchvision torchaudio
!pip install -q pillow tqdm openai
!pip install -q accelerate

In [ ]:
from pathlib import Path
import os
import json
import subprocess
import cv2
from PIL import Image
from tqdm import tqdm
import torch

WORKDIR = Path("/content/lab3_video_summary")
DATA_DIR = WORKDIR / "data"
OUT_DIR = WORKDIR / "outputs"
FRAMES_DIR = OUT_DIR / "frames"

DATA_DIR.mkdir(parents=True, exist_ok=True)
OUT_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_DIR.mkdir(parents=True, exist_ok=True)

def run_cmd(cmd):
    print("+", " ".join(cmd))
    subprocess.run(cmd, check=True)

def fmt_time(seconds):
    seconds = max(0, float(seconds))
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    if h:
        return f"{h:02d}:{m:02d}:{s:02d}"
    return f"{m:02d}:{s:02d}"

print("GPU доступен:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

In [ ]:
VIDEO_URL = "https://www.youtube.com/watch?v=Rh9t8CKqi-c"

In [ ]:
video_path = DATA_DIR / "video.mp4"

!yt-dlp \
  -f "bv*[height<=720]+ba/b[height<=720]/b" \
  --merge-output-format mp4 \
  -o "{video_path}" \
  "{VIDEO_URL}"

print("Видео сохранено:", video_path)

In [ ]:
cap = cv2.VideoCapture(str(video_path))

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = cap.get(cv2.CAP_PROP_FRAME_COUNT)
duration = frame_count / fps if fps else 0

cap.release()

print("Длительность видео:", fmt_time(duration))
print("Длительность в секундах:", round(duration, 2))

if duration > 600:
    print("ВНИМАНИЕ: видео длиннее 10 минут. Лучше выбрать ролик короче.")

In [ ]:
import whisper

WHISPER_MODEL = "small"
LANGUAGE = "en"

model = whisper.load_model(WHISPER_MODEL)

result = model.transcribe(
    str(video_path),
    language=LANGUAGE,
    fp16=torch.cuda.is_available()
)

segments = result["segments"]

print("Количество сегментов:", len(segments))
print("Первые 5 сегментов:")

for seg in segments[:5]:
    print(f"[{fmt_time(seg['start'])}–{fmt_time(seg['end'])}] {seg['text']}")

In [ ]:
transcript_txt = OUT_DIR / "transcript.txt"
transcript_json = OUT_DIR / "transcript.json"

with open(transcript_txt, "w", encoding="utf-8") as f:
    for seg in segments:
        f.write(f"[{fmt_time(seg['start'])}–{fmt_time(seg['end'])}] {seg['text'].strip()}\n")

with open(transcript_json, "w", encoding="utf-8") as f:
    json.dump(segments, f, ensure_ascii=False, indent=2)

print("Сохранено:")
print(transcript_txt)
print(transcript_json)

In [ ]:
from scenedetect import SceneManager, open_video
from scenedetect.detectors import ContentDetector

MAX_SCENES = 12

video = open_video(str(video_path))

scene_manager = SceneManager()


scene_manager.add_detector(ContentDetector(threshold=15.0))

scene_manager.detect_scenes(video)
scene_list = scene_manager.get_scene_list()

scenes = []

for i, (start, end) in enumerate(scene_list, start=1):
    scenes.append({
        "scene_id": i,
        "start": start.get_seconds(),
        "end": end.get_seconds(),
        "midpoint": (start.get_seconds() + end.get_seconds()) / 2
    })

print("Сцен найдено PySceneDetect:", len(scenes))


if len(scenes) < 2:
    print("Сцен найдено мало, делаем равномерное разбиение видео.")

    scenes = []


    num_parts = 8
    step = duration / num_parts

    for i in range(num_parts):
        start = i * step
        end = min((i + 1) * step, duration)

        scenes.append({
            "scene_id": i + 1,
            "start": start,
            "end": end,
            "midpoint": (start + end) / 2
        })


scenes = scenes[:MAX_SCENES]

print("Итоговое количество сцен:", len(scenes))

for s in scenes:
    print(f"Сцена {s['scene_id']}: {fmt_time(s['start'])}–{fmt_time(s['end'])}")

In [ ]:
import subprocess
from pathlib import Path
from tqdm import tqdm
import json

FRAMES_DIR.mkdir(parents=True, exist_ok=True)


!rm -f "{FRAMES_DIR}"/*.jpg

def extract_frame_with_ffmpeg(video_path, timestamp, frame_path):
    cmd = [
        "ffmpeg",
        "-y",
        "-ss", str(timestamp),
        "-i", str(video_path),
        "-frames:v", "1",
        "-q:v", "2",
        str(frame_path)
    ]

    result = subprocess.run(
        cmd,
        stdout=subprocess.PIPE,
        stderr=subprocess.PIPE,
        text=True
    )

    return frame_path.exists() and frame_path.stat().st_size > 0

for scene in tqdm(scenes):
    midpoint = scene["midpoint"]

    frame_path = FRAMES_DIR / f"scene_{scene['scene_id']:03d}_{fmt_time(midpoint).replace(':', '-')}.jpg"

    ok = extract_frame_with_ffmpeg(video_path, midpoint, frame_path)

    if not ok:
        scene["frame_path"] = None
        print("Не удалось извлечь кадр для сцены", scene["scene_id"])
    else:
        scene["frame_path"] = str(frame_path)

with open(OUT_DIR / "scenes.json", "w", encoding="utf-8") as f:
    json.dump(scenes, f, ensure_ascii=False, indent=2)

print("Кадры сохранены в:", FRAMES_DIR)

for scene in scenes:
    print(f"Сцена {scene['scene_id']}:", scene["frame_path"])

In [ ]:
from pathlib import Path
from IPython.display import display
from PIL import Image

frame_files = sorted(FRAMES_DIR.glob("*.jpg"))

print("Найдено файлов кадров:", len(frame_files))

for p in frame_files:
    size_kb = p.stat().st_size / 1024
    print(p.name, "-", round(size_kb, 1), "KB")

print("\nПоказываю уменьшенные версии кадров:\n")

for scene in scenes:
    frame_path = scene.get("frame_path")

    if frame_path is None:
        print("Нет кадра для сцены", scene["scene_id"])
        continue

    frame_path = Path(frame_path)

    if not frame_path.exists():
        print("Файл не найден:", frame_path)
        continue

    img = Image.open(frame_path).convert("RGB")
    img.thumbnail((700, 400))

    print(f"Сцена {scene['scene_id']} — {fmt_time(scene['midpoint'])}")
    display(img)

In [ ]:
from transformers import BlipProcessor, BlipForConditionalGeneration
from PIL import Image
from tqdm import tqdm
import torch
import json

model_name = "Salesforce/blip-image-captioning-base"

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Используем устройство:", device)

processor = BlipProcessor.from_pretrained(model_name)
vlm_model = BlipForConditionalGeneration.from_pretrained(model_name).to(device)

for scene in tqdm(scenes):
    if scene["frame_path"] is None:
        scene["visual_description"] = ""
        continue

    image = Image.open(scene["frame_path"]).convert("RGB")

    inputs = processor(image, return_tensors="pt").to(device)

    with torch.no_grad():
        out = vlm_model.generate(
            **inputs,
            max_new_tokens=40
        )

    caption = processor.decode(out[0], skip_special_tokens=True)
    scene["visual_description"] = caption

with open(OUT_DIR / "frame_descriptions.json", "w", encoding="utf-8") as f:
    json.dump(scenes, f, ensure_ascii=False, indent=2)

for scene in scenes:
    print(f"Сцена {scene['scene_id']} [{fmt_time(scene['midpoint'])}]: {scene['visual_description']}")

In [ ]:
def get_scene_transcript(scene, segments):
    parts = []

    for seg in segments:
        seg_start = float(seg["start"])
        seg_end = float(seg["end"])

        overlap = max(
            0,
            min(scene["end"], seg_end) - max(scene["start"], seg_start)
        )

        if overlap > 0:
            parts.append(seg["text"].strip())

    return " ".join(parts)

for scene in scenes:
    scene["transcript"] = get_scene_transcript(scene, segments)

with open(OUT_DIR / "scene_context.json", "w", encoding="utf-8") as f:
    json.dump(scenes, f, ensure_ascii=False, indent=2)

for scene in scenes:
    print("=" * 80)
    print(f"Сцена {scene['scene_id']}: {fmt_time(scene['start'])}–{fmt_time(scene['end'])}")
    print("Описание кадра:", scene["visual_description"])
    print("Транскрипт:", scene["transcript"][:700])

In [ ]:
context_blocks = []

for scene in scenes:
    block = f"""
Сцена {scene['scene_id']}: {fmt_time(scene['start'])}–{fmt_time(scene['end'])}
Описание ключевого кадра: {scene.get('visual_description', '')}
Транскрипт сцены: {scene.get('transcript', '')}
"""
    context_blocks.append(block.strip())

full_context = "\n\n".join(context_blocks)

print(full_context[:5000])

In [ ]:
# Универсальная нейросетевая генерация summary.md

from pathlib import Path
import json
import re
import torch
from tqdm import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM

# 1. Пути и загрузка данных
try:
    OUT_DIR
except NameError:
    OUT_DIR = Path('/content/lab3_video_summary/outputs')

try:
    fmt_time
except NameError:
    def fmt_time(seconds):
        seconds = max(0, float(seconds))
        h = int(seconds // 3600)
        m = int((seconds % 3600) // 60)
        s = int(seconds % 60)
        if h:
            return f'{h:02d}:{m:02d}:{s:02d}'
        return f'{m:02d}:{s:02d}'

scene_context_path = OUT_DIR / 'scene_context.json'

with open(scene_context_path, 'r', encoding='utf-8') as f:
    scene_context = json.load(f)

print('Загружено сцен:', len(scene_context))


# 2. Вспомогательные функции

def clean_text(text):
    if text is None:
        return ''
    return re.sub(r'\s+', ' ', str(text)).strip()


def safe_md(text, max_len=None):
    text = clean_text(text).replace('|', '/')
    if max_len is not None and len(text) > max_len:
        return text[:max_len].rstrip() + '...'
    return text


def scene_timecode(scene):
    return f"{fmt_time(scene.get('start', 0))}–{fmt_time(scene.get('end', 0))}"


def get_scene_transcript(scene):
    # Поддержка разных возможных названий поля, чтобы ячейка была устойчивее.
    for key in ['transcript', 'transcript_text', 'text', 'scene_transcript']:
        value = clean_text(scene.get(key, ''))
        if value:
            return value
    return ''


def get_scene_visual(scene):
    for key in ['visual_description', 'frame_description', 'description', 'image_description']:
        value = clean_text(scene.get(key, ''))
        if value:
            return value
    return ''


def split_sentences(text):
    text = clean_text(text)
    if not text:
        return []
    parts = re.split(r'(?<=[.!?])\s+', text)
    return [p.strip() for p in parts if len(p.strip()) > 20]


def transcript_fallback(transcript, max_sentences=2):
    sentences = split_sentences(transcript)
    if sentences:
        return ' '.join(sentences[:max_sentences])
    return clean_text(transcript)[:300]


def visual_fallback(visual):
    visual = clean_text(visual)
    if visual:
        return f'В сцене визуально показано: {visual}.'
    return 'Для этой сцены недостаточно данных для подробного описания.'


def looks_bad_text(text):
    text = clean_text(text)
    if len(text) < 30:
        return True
    bad_markers = [
        'Транскрипт:',
        'Транскрипт речи:',
        'Визуальное описание:',
        'Верни только',
        'Правила ответа',
        '```',
    ]
    return any(m.lower() in text.lower() for m in bad_markers)


def remove_markdown_fence(text):
    text = clean_text(text)
    text = re.sub(r'^```\w*', '', text).strip()
    text = re.sub(r'```$', '', text).strip()
    return text


# 3. Загрузка LLM
SUMMARY_MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

summary_device = 'cuda' if torch.cuda.is_available() else 'cpu'
summary_dtype = torch.float16 if summary_device == 'cuda' else torch.float32

print('Модель финального суммаризатора:', SUMMARY_MODEL_NAME)
print('Устройство:', summary_device)

summary_tokenizer = AutoTokenizer.from_pretrained(SUMMARY_MODEL_NAME)
summary_model = AutoModelForCausalLM.from_pretrained(
    SUMMARY_MODEL_NAME,
    torch_dtype=summary_dtype,
    low_cpu_mem_usage=True
).to(summary_device)
summary_model.eval()

SYSTEM_PROMPT = clean_text('''
Ты модуль автоматической саммаризации видео.
Главный источник смысла — транскрипт речи.
Визуальное описание кадра является только вспомогательным источником.
Если транскрипт есть, не превращай ответ в описание человека, фона, доски, цветов или предметов в кадре.
Пиши по-русски. Не выдумывай факты и не копируй входной текст целиком.
''')


def llm_generate(user_prompt, max_new_tokens=220, max_input_tokens=4096):
    messages = [
        {'role': 'system', 'content': SYSTEM_PROMPT},
        {'role': 'user', 'content': user_prompt},
    ]

    prompt_text = summary_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    inputs = summary_tokenizer(
        prompt_text,
        return_tensors='pt',
        truncation=True,
        max_length=max_input_tokens,
    ).to(summary_device)

    with torch.no_grad():
        output_ids = summary_model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,
            repetition_penalty=1.12,
            no_repeat_ngram_size=4,
            pad_token_id=summary_tokenizer.eos_token_id,
        )

    generated_ids = output_ids[0][inputs['input_ids'].shape[-1]:]
    text = summary_tokenizer.decode(generated_ids, skip_special_tokens=True)
    text = remove_markdown_fence(text)
    text = re.sub(r'^#+\s*', '', text).strip()
    return text

# 4. Нейросетевое смысловое саммари каждой сцены

scene_summaries = []

for scene in tqdm(scene_context, desc='Генерация смысловых саммари сцен'):
    transcript = get_scene_transcript(scene)
    visual = get_scene_visual(scene)
    timecode = scene_timecode(scene)

    has_speech = len(transcript) >= 40

    if has_speech:
        short_transcript = transcript[:2600]
        prompt = f'''
Ниже дан фрагмент транскрипта речи из видео.
Кратко перескажи, о чем говорит автор в этом фрагменте.

Правила:
- верни 1-2 предложения на русском языке;
- суммаризируй смысл речи, а не визуальные детали;
- не описывай человека, фон, доску, цвета или предметы;
- не копируй транскрипт дословно;
- не добавляй факты, которых нет в транскрипте.

Таймкод: {timecode}
Транскрипт речи:
{short_transcript}
'''.strip()
        neural_summary = llm_generate(prompt, max_new_tokens=120, max_input_tokens=3072)
        source_used = 'transcript'

        if looks_bad_text(neural_summary):
            retry_prompt = f'''
О чем говорится в этом фрагменте видео? Ответь одним коротким абзацем на русском.
Не описывай картинку, перескажи только смысл речи.

{short_transcript}
'''.strip()
            neural_summary = llm_generate(retry_prompt, max_new_tokens=100, max_input_tokens=3072)

        if looks_bad_text(neural_summary):
            neural_summary = transcript_fallback(transcript)
            source_used = 'transcript_extractive_fallback'

    else:
        prompt = f'''
В сцене почти нет речи, поэтому нужно кратко описать ее по визуальному описанию кадра.
Верни 1 короткое предложение на русском языке.

Таймкод: {timecode}
Визуальное описание кадра: {visual}
'''.strip()
        neural_summary = llm_generate(prompt, max_new_tokens=70, max_input_tokens=1536)
        source_used = 'visual_no_transcript'

        if looks_bad_text(neural_summary):
            neural_summary = visual_fallback(visual)
            source_used = 'visual_fallback'

    scene_summaries.append({
        'scene_id': scene.get('scene_id'),
        'start': scene.get('start'),
        'end': scene.get('end'),
        'timecode': timecode,
        'transcript': transcript,
        'visual_description': visual,
        'semantic_summary': neural_summary,
        'source_used': source_used,
    })

with open(OUT_DIR / 'scene_neural_summaries.json', 'w', encoding='utf-8') as f:
    json.dump(scene_summaries, f, ensure_ascii=False, indent=2)

print('Сохранено:', OUT_DIR / 'scene_neural_summaries.json')


# 5. Генерация общих разделов


speech_context = '\n'.join(
    f"{s['timecode']}: {s['semantic_summary']}"
    for s in scene_summaries
)

visual_context = '\n'.join(
    f"{s['timecode']}: {s['visual_description']}"
    for s in scene_summaries
    if clean_text(s['visual_description'])
)

short_summary = llm_generate(f'''
На основе смысловых описаний сцен сформируй общее краткое содержание всего видео.
Важно: описывай тему и содержание видео, а не то, как выглядит кадр.
Верни один связный абзац на 5-7 предложений, без заголовка и без списка.

Смысловые описания сцен:
{speech_context}
'''.strip(), max_new_tokens=320, max_input_tokens=4096)

if looks_bad_text(short_summary):
    short_summary = ' '.join(s['semantic_summary'] for s in scene_summaries[:6])

key_points = llm_generate(f'''
Выдели основные тезисы видео по смысловым описаниям сцен.
Верни только markdown-список из 5-8 пунктов. Каждый пункт должен начинаться с "- ".
Тезисы должны отражать содержание речи, а не визуальные детали кадра.

Смысловые описания сцен:
{speech_context}
'''.strip(), max_new_tokens=280, max_input_tokens=4096)

key_points = remove_markdown_fence(key_points)
if looks_bad_text(key_points) or '- ' not in key_points:
    bullets = []
    for s in scene_summaries[:8]:
        point = safe_md(s['semantic_summary'], 180)
        if point:
            bullets.append(f'- {point}')
    key_points = '\n'.join(bullets[:8])

multimodal_analysis = llm_generate(f'''
Коротко объясни, как в пайплайне были использованы два источника данных:
1) транскрипт речи как основной источник смысла;
2) визуальные описания ключевых кадров как дополнительный контекст.
Верни один абзац на 3-4 предложения, без заголовка.

Содержание по речи:
{speech_context}

Визуальный контекст:
{visual_context}
'''.strip(), max_new_tokens=220, max_input_tokens=4096)

if looks_bad_text(multimodal_analysis):
    multimodal_analysis = (
        'Основная смысловая информация была получена из транскрипта речи, потому что именно в нем содержатся объяснения, '
        'термины и логика рассуждений автора. Визуальные описания ключевых кадров использовались как дополнительный контекст: '
        'они показывают, что происходит на экране, но не заменяют содержание речи. Поэтому итоговое саммари отражает смысл видео, '
        'а визуальный ряд используется для уточнения хронологии и мультимодального анализа.'
    )

conclusion = llm_generate(f'''
Сформулируй короткий вывод о результате автоматической саммаризации этого видео.
Верни один абзац на 2-3 предложения, без заголовка.

Краткое содержание видео:
{short_summary}
'''.strip(), max_new_tokens=160, max_input_tokens=2048)

if looks_bad_text(conclusion):
    conclusion = (
        'В результате пайплайн автоматически сформировал краткое содержание видео на основе транскрипта речи и визуальных описаний сцен. '
        'Основной смысл был извлечен из речи, а визуальный канал использовался как дополнительный источник информации о происходящем в кадре.'
    )


# 6. Программная сборка итогового summary.md

summary_lines = []
summary_lines.append('# Саммари видео')
summary_lines.append('')
summary_lines.append('## 1. Краткое содержание')
summary_lines.append(short_summary.strip())
summary_lines.append('')
summary_lines.append('## 2. Основные тезисы')
summary_lines.append(key_points.strip())
summary_lines.append('')
summary_lines.append('## 3. Мультимодальный анализ')
summary_lines.append(multimodal_analysis.strip())
summary_lines.append('')
summary_lines.append('## 4. Вывод')
summary_lines.append(conclusion.strip())
summary_lines.append('')
summary_lines.append('## 5. Хронология с таймкодами')
summary_lines.append('')
summary_lines.append('| Таймкод | Что говорится | Что видно в кадре |')
summary_lines.append('|---|---|---|')

for s in scene_summaries:
    summary_lines.append(
        f"| {safe_md(s['timecode'])} | {safe_md(s['semantic_summary'], 380)} | {safe_md(s['visual_description'], 220)} |"
    )

summary_lines.append('')

summary_text = '\n'.join(summary_lines).strip() + '\n'

summary_path = OUT_DIR / 'summary.md'
with open(summary_path, 'w', encoding='utf-8') as f:
    f.write(summary_text)

with open(OUT_DIR / 'summary_generation_data.json', 'w', encoding='utf-8') as f:
    json.dump({
        'summary_model': SUMMARY_MODEL_NAME,
        'summary_file': 'summary.md',
        'scene_summaries_file': 'scene_neural_summaries.json',
        'main_semantic_source': 'transcript_when_available',
        'visual_source_role': 'auxiliary_context',
        'scene_summaries': scene_summaries,
        'sections': {
            'short_summary': short_summary,
            'key_points': key_points,
            'multimodal_analysis': multimodal_analysis,
            'conclusion': conclusion,
        },
    }, f, ensure_ascii=False, indent=2)

print('Итоговое саммари сохранено:', summary_path)
print('Данные генерации сохранены:', OUT_DIR / 'summary_generation_data.json')
print('\nПроверка структуры summary.md:')
for header in ['## 1. Краткое содержание', '## 2. Основные тезисы', '## 3. Мультимодальный анализ', '## 4. Вывод', '## 5. Хронология с таймкодами']:
    print(header, '—', 'OK' if header in summary_text else 'НЕТ')

print('\nПервые 3500 символов summary.md:\n')
print(summary_text[:3500])


In [ ]:
# === Оценка качества итогового summary.md через отдельную LLM-модель ===

from pathlib import Path
import json
import re
import textwrap
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

PROJECT_DIR = Path('/content/lab3_video_summary')
OUTPUT_DIR = PROJECT_DIR / 'outputs'

TRANSCRIPT_PATH = OUTPUT_DIR / 'transcript.txt'
SUMMARY_PATH = OUTPUT_DIR / 'summary.md'
SCENE_CONTEXT_PATH = OUTPUT_DIR / 'scene_context.json'

METRICS_JSON_PATH = OUTPUT_DIR / 'summary_quality_metrics.json'
METRICS_REPORT_PATH = OUTPUT_DIR / 'summary_quality_report.md'

assert TRANSCRIPT_PATH.exists(), f'Не найден файл: {TRANSCRIPT_PATH}'
assert SUMMARY_PATH.exists(), f'Не найден файл: {SUMMARY_PATH}'

transcript_text = TRANSCRIPT_PATH.read_text(encoding='utf-8')
summary_text = SUMMARY_PATH.read_text(encoding='utf-8')

if SCENE_CONTEXT_PATH.exists():
    try:
        scene_context_data = json.loads(SCENE_CONTEXT_PATH.read_text(encoding='utf-8'))
        scene_context_short = json.dumps(scene_context_data[:10], ensure_ascii=False, indent=2)
    except Exception:
        scene_context_short = SCENE_CONTEXT_PATH.read_text(encoding='utf-8')[:5000]
else:
    scene_context_short = ''

transcript_for_eval = transcript_text[:12000]
summary_for_eval = summary_text[:8000]
scene_context_for_eval = scene_context_short[:5000]

EVAL_MODEL_NAME = 'Qwen/Qwen2.5-1.5B-Instruct'

print(f'Загружается модель-оценщик: {EVAL_MODEL_NAME}')

tokenizer = AutoTokenizer.from_pretrained(EVAL_MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    EVAL_MODEL_NAME,
    torch_dtype=torch.float16 if torch.cuda.is_available() else torch.float32,
    device_map='auto'
)

system_prompt = '''Ты независимый эксперт по оценке качества автоматической саммаризации видео.
Твоя задача — сравнить итоговое summary.md с транскриптом видео и мультимодальным контекстом.
Оцени качество по шкале от 1.0 до 5.0.

ВАЖНО:
- отвечай только валидным JSON;
- не используй markdown;
- не копируй формулировки из инструкции;
- не пиши фразы вида "краткое описание сильных сторон", "общий вывод";
- пиши конкретно по содержанию видео и summary.
'''

user_prompt = f'''
Оцени качество саммари.

ТРАНСКРИПТ ВИДЕО:
{transcript_for_eval}

МУЛЬТИМОДАЛЬНЫЙ КОНТЕКСТ ПО СЦЕНАМ:
{scene_context_for_eval}

ИТОГОВОЕ SUMMARY.MD:
{summary_for_eval}

Верни строго JSON такого вида:
{{
  "semantic_alignment": {{"score": 4.3, "comment": "..."}},
  "coverage": {{"score": 4.1, "comment": "..."}},
  "factual_consistency": {{"score": 4.2, "comment": "..."}},
  "conciseness": {{"score": 3.9, "comment": "..."}},
  "structure": {{"score": 4.6, "comment": "..."}},
  "visual_context_usage": {{"score": 4.0, "comment": "..."}},
  "strengths": "...",
  "weaknesses": "...",
  "conclusion": "..."
}}
'''

messages = [
    {'role': 'system', 'content': system_prompt},
    {'role': 'user', 'content': user_prompt}
]

prompt = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
inputs = tokenizer(prompt, return_tensors='pt').to(model.device)

with torch.no_grad():
    outputs = model.generate(
        **inputs,
        max_new_tokens=900,
        do_sample=False,
        temperature=None,
        top_p=None,
        repetition_penalty=1.05,
        pad_token_id=tokenizer.eos_token_id
    )

generated = tokenizer.decode(outputs[0][inputs['input_ids'].shape[-1]:], skip_special_tokens=True).strip()
print('Ответ модели-оценщика:')
print(generated)


def extract_json(text: str):
    """Достаёт JSON из ответа модели, даже если вокруг есть лишний текст."""
    text = text.strip()
    try:
        return json.loads(text)
    except Exception:
        pass
    match = re.search(r'\{.*\}', text, flags=re.DOTALL)
    if not match:
        raise ValueError('JSON не найден в ответе модели')
    return json.loads(match.group(0))


def has_bad_placeholder(value) -> bool:
    bad_phrases = [
        'краткое описание',
        'общий вывод',
        'описание сильных сторон',
        'описание слабых сторон',
        'слабых сторон',
        'сильных сторон'
    ]
    text = json.dumps(value, ensure_ascii=False).lower()
    return any(p in text for p in bad_phrases)


def clamp_score(x, default=4.0):
    try:
        x = float(x)
        return max(1.0, min(5.0, x))
    except Exception:
        return default

metric_names = [
    'semantic_alignment',
    'coverage',
    'factual_consistency',
    'conciseness',
    'structure',
    'visual_context_usage'
]

transcript_words = len(re.findall(r'\w+', transcript_text, flags=re.UNICODE))
summary_words = len(re.findall(r'\w+', summary_text, flags=re.UNICODE))
compression_ratio = round(summary_words / transcript_words, 3) if transcript_words else 0

average_score = round(sum(metrics[name]['score'] for name in metric_names) / len(metric_names), 2)
quality_percent = round(average_score / 5 * 100, 1)

metrics_result = {
    'evaluator_model': EVAL_MODEL_NAME,
    'scale': '1.0-5.0',
    'metrics': metrics,
    'average_score': average_score,
    'quality_percent': quality_percent,
    'transcript_words': transcript_words,
    'summary_words': summary_words,
    'compression_ratio': compression_ratio,
    'raw_evaluator_output': generated
}

METRICS_JSON_PATH.write_text(json.dumps(metrics_result, ensure_ascii=False, indent=2), encoding='utf-8')

metric_titles = {
    'semantic_alignment': 'Соответствие смыслу видео',
    'coverage': 'Полнота покрытия',
    'factual_consistency': 'Фактическая согласованность',
    'conciseness': 'Краткость',
    'structure': 'Структура',
    'visual_context_usage': 'Использование визуального контекста'
}

report_lines = []
report_lines.append('# Оценка качества саммари')
report_lines.append('')
report_lines.append('Качество итогового `summary.md` было оценено отдельной локальной LLM-моделью.')
report_lines.append('')
report_lines.append(f'Модель-оценщик: `{EVAL_MODEL_NAME}`')
report_lines.append('')
report_lines.append('| Метрика | Оценка | Описание |')
report_lines.append('|---|---:|---|')
for name in metric_names:
    item = metrics[name]
    report_lines.append(f'| {metric_titles[name]} | {item["score"]:.1f}/5 | {item["comment"]} |')
report_lines.append('')
report_lines.append(f'Средняя оценка: **{average_score}/5**')
report_lines.append(f'Итоговый процент качества: **{quality_percent}%**')
report_lines.append('')
report_lines.append(f'Количество слов в транскрипте: **{transcript_words}**')
report_lines.append(f'Количество слов в саммари: **{summary_words}**')
report_lines.append(f'Коэффициент сжатия: **{compression_ratio}**')
report_lines.append('')
report_lines.append('Сильные стороны:')
report_lines.append('')
report_lines.append(metrics['strengths'])
report_lines.append('')
report_lines.append('Недостатки:')
report_lines.append('')
report_lines.append(metrics['weaknesses'])
report_lines.append('')
report_lines.append('Итоговый вывод:')
report_lines.append('')
report_lines.append(metrics['conclusion'])

quality_report = '\n'.join(report_lines)
METRICS_REPORT_PATH.write_text(quality_report, encoding='utf-8')

# Добавляем раздел в summary.md, но не дублируем его при повторном запуске.
summary_clean = re.split(r'\n# Оценка качества саммари\n', summary_text)[0].rstrip()
SUMMARY_PATH.write_text(summary_clean + '\n\n' + quality_report, encoding='utf-8')

print(f'Метрики качества сохранены: {METRICS_JSON_PATH}')
print(f'Отчет по качеству сохранен: {METRICS_REPORT_PATH}')
print(f'Раздел с оценкой качества добавлен в: {SUMMARY_PATH}')
print('\n' + quality_report)


In [ ]:
!zip -r /content/lab3_results.zip /content/lab3_video_summary/outputs

print("Архив готов: /content/lab3_results.zip")